# GPT 4.1 mini

# Severity Verification, Detection Controls verification , DFMEA PFMEA gap analysis



# severity updated

In [2]:
import os
import re
import copy
import time
import webbrowser
import openpyxl
import pandas as pd
from openpyxl.styles import Alignment, Font
from openpyxl.worksheet.cell_range import CellRange
from dotenv import load_dotenv
from openai import OpenAI
from xlsx2html import xlsx2html
from openpyxl.utils import get_column_letter

# Load the variables from .env into the environment
load_dotenv() 

# Initialize the LLM Client with your custom configuration
client = OpenAI(
    api_key=os.getenv("API_KEY"),
    base_url="https://api.euron.one/api/v1/euri",
)

def extract_number(val):
    """Safely extracts the first integer from a string or number without using an LLM."""
    if val is None:
        return None
    if isinstance(val, (int, float)):
        return int(val)
    matches = re.findall(r'\d+', str(val))
    return int(matches[0]) if matches else None

def get_det_zone(sev, det):
    """Maps Severity and Detection to a Detection Zone (1, 2, or 3) based on the standard matrix."""
    if sev is None or det is None:
        return ""
    try:
        s, d = int(sev), int(det)
    except ValueError:
        return ""
    if not (1 <= s <= 10 and 1 <= d <= 10):
        return ""
        
    matrix = {
        10: {1:3, 2:2, 3:1, 4:1, 5:1, 6:1, 7:1, 8:1, 9:1, 10:1},
        9:  {1:3, 2:2, 3:1, 4:1, 5:1, 6:1, 7:1, 8:1, 9:1, 10:1},
        8:  {1:3, 2:2, 3:2, 4:2, 5:2, 6:2, 7:1, 8:1, 9:1, 10:1},
        7:  {1:3, 2:3, 3:3, 4:2, 5:2, 6:2, 7:2, 8:1, 9:1, 10:1},
        6:  {1:3, 2:3, 3:3, 4:3, 5:3, 6:2, 7:2, 8:1, 9:1, 10:1},
        5:  {1:3, 2:3, 3:3, 4:3, 5:3, 6:3, 7:3, 8:2, 9:2, 10:2},
        4:  {1:3, 2:3, 3:3, 4:3, 5:3, 6:3, 7:3, 8:2, 9:2, 10:2},
        3:  {1:3, 2:3, 3:3, 4:3, 5:3, 6:3, 7:3, 8:3, 9:3, 10:3},
        2:  {1:3, 2:3, 3:3, 4:3, 5:3, 6:3, 7:3, 8:3, 9:3, 10:3},
        1:  {1:3, 2:3, 3:3, 4:3, 5:3, 6:3, 7:3, 8:3, 9:3, 10:3}
    }
    return matrix[d][s]

def get_sev_zone(sev, occ):
    """Maps Severity and Occurrence to a Severity Zone (1, 2, or 3) based on the image matrix."""
    if sev is None or occ is None:
        return ""
    try:
        s, o = int(sev), int(occ)
    except ValueError:
        return ""
    if not (1 <= s <= 10 and 1 <= o <= 10):
        return ""
        
    matrix = {
        10: {1:3, 2:1, 3:1, 4:1, 5:1, 6:1, 7:1, 8:1, 9:1, 10:1},
        9:  {1:3, 2:1, 3:1, 4:1, 5:1, 6:1, 7:1, 8:1, 9:1, 10:1},
        8:  {1:3, 2:2, 3:1, 4:1, 5:1, 6:1, 7:1, 8:1, 9:1, 10:1},
        7:  {1:3, 2:2, 3:2, 4:2, 5:1, 6:1, 7:1, 8:1, 9:1, 10:1},
        6:  {1:3, 2:2, 3:2, 4:2, 5:1, 6:1, 7:1, 8:1, 9:1, 10:1},
        5:  {1:3, 2:3, 3:2, 4:2, 5:2, 6:2, 7:1, 8:1, 9:1, 10:1},
        4:  {1:3, 2:3, 3:3, 4:3, 5:2, 6:2, 7:1, 8:1, 9:1, 10:1},
        3:  {1:3, 2:3, 3:3, 4:3, 5:3, 6:3, 7:2, 8:2, 9:1, 10:1},
        2:  {1:3, 2:3, 3:3, 4:3, 5:3, 6:3, 7:2, 8:2, 9:1, 10:1},
        1:  {1:3, 2:3, 3:3, 4:3, 5:3, 6:3, 7:3, 8:3, 9:3, 10:3}
    }
    return matrix[o][s]

def evaluate_severity_with_llm(effect_text, original_sev):
    if not effect_text or str(effect_text).strip().lower() == 'nan':
        return "", "", 0, 0

    try:
        clean_original_sev = int(float(original_sev))
    except (ValueError, TypeError):
        clean_original_sev = str(original_sev).strip()

    system_prompt = """
    You are an expert AIAG PFMEA auditor. Your strict task is to determine the TRUE Severity score based ONLY on the English text describing the failure effect.

    AIAG SEVERITY DEFINITIONS:
    10: Product: Affects safe operation and/or involves noncompliance with regulations without warning. Process: May endanger operator, machine or assembly without warning.
    9: Product: Affects safe operation and/or involves noncompliance with regulations with warning. Process: May endanger operator, machine or assembly with warning.
    8: Product: Loss of primary function (product inoperable, does not affect safe operation). Process: 100% of product may have to be scrapped. Line shutdown or stop ship.
    7: Product: Degradation of primary function (product operable, but at a reduced level of performance). Process: A portion of the production run may have to be scrapped. Deviation from primary process; decreased line speed or added manpower.
    6: Product: Loss of secondary function (product operable but service life greatly reduced, convenience item(s) inoperable, customer dissatisfied). Process: 100% of production run may have to be reworked off line and accepted.
    5: Product: Degradation of secondary function (product operable but appearance affected, convenience item(s) operable at a reduced level, customer dissatisfied. Process: A proportion of the production run may have to be reworked off line and accepted.
    4: Product: Appearance, fit and finish type items do not conform, defect noticed by most of the customers (>75%). Process: 100% of production run may have to be reworked in station before it is processed.
    3: Product: Appearance, fit and finish type items do not conform, defect noticed by about half of the customers (50%). Process: A proportion of the production run may have to be reworked in station before it is processed.
    2: Product: Appearance, fit and finish type items do not conform, defect noticed by discriminating customers (<25%). Process: Slight inconvenience to process, operation or operator.
    1: Product: No discernible effect. Process: No discernible effect.

    RULES:
    1. Ignore any numbers written in parentheses (e.g., "(4)") inside the text when determining the TRUE severity.
    2. If multiple effects are described, the TRUE severity is the HIGHEST AIAG number described.
    3. Compare your TRUE severity to the provided "Stated Severity" variable.
    4. Output your response EXACTLY in this format on two lines:
    Severity: <integer 1-10>
    Reason: <If your TRUE severity matches the Stated Severity, write "severity is correct". If it does not match, explain exactly why.>

    LEARNING EXAMPLES:
    Text: "S: Portion of production run may have to be scrapped (5) \n OEM: Portion of production run may have to be scrapped (7)"
    Stated Severity: 7
    Output:
    Severity: 7
    Reason: severity is correct

    Text: "MismatchEU: Loss of primary function (product inoperable, does not affect safe operation) (4)"
    Stated Severity: 4
    Output:
    Severity: 8
    Reason: 'Loss of primary function' maps to AIAG 8. The stated (4) is an under-rating.

    Text: "S: A proportion of the production run may have to be reworked off line and accepted (5)\nOEM: No discernible effect (1)"
    Stated Severity: 5
    Output:
    Severity: 5
    Reason: severity is correct
    """

    try:
        response = client.chat.completions.create(
            model="gpt-5.4-mini",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": f"Stated Severity: {clean_original_sev}\nText: {effect_text}\nOutput:"}
            ],
            temperature=0.0
        )
        
        output_text = response.choices[0].message.content.strip()
        severity, reason = "", ""
        for line in output_text.split('\n'):
            if line.startswith("Severity:"):
                severity = line.replace("Severity:", "").strip()
            elif line.startswith("Reason:"):
                reason = line.replace("Reason:", "").strip()
        return severity, reason, response.usage.prompt_tokens, response.usage.completion_tokens
    except Exception as e:
        return "", f"Error: {str(e)}", 0, 0

def build_det_prompt(file_path):
    xls = pd.ExcelFile(file_path)
    if "P-DET" in xls.sheet_names:
        df = pd.read_excel(xls, "P-DET")
    elif "p-det" in xls.sheet_names:
        df = pd.read_excel(xls, "p-det")
    else:
        return "AIAG DETECTION DEFINITIONS NOT FOUND IN EXCEL."
    
    prompt = "AIAG DETECTION DEFINITIONS:\n"
    for i in range(len(df)):
        rank = str(df.iloc[i, 1]).strip()
        cat = str(df.iloc[i, 2]).strip()
        crit = str(df.iloc[i, 3]).strip()
        if rank.isdigit():
            prompt += f"- {rank} ({cat}): {crit}\n"
    return prompt

def extract_strongest_detection(det_text):
    if not det_text:
        return None
    lowest_val = 999
    matches = re.findall(r"\((\d+)\)", str(det_text))
    if not matches:
        matches = re.findall(r"\b(\d+)\b", str(det_text))
        
    if matches:
        for m in matches:
            val = int(m)
            if val < lowest_val:
                lowest_val = val
    return lowest_val if lowest_val != 999 else None

def evaluate_detection_with_llm(full_det_text, original_det, pred_det, det_definitions):
    if not full_det_text:
        return "", 0, 0
    
    try:
        clean_original_det = int(float(original_det))
    except (ValueError, TypeError):
        clean_original_det = str(original_det).strip()
        
    system_prompt = f"""
You are an expert AIAG PFMEA auditor. Your job is to strictly verify the consistency of the 'Current Process Detection Controls' column against the assigned Detection rankings.

{det_definitions}

INSTRUCTIONS FOR AUDIT:
I will provide you with:
1. Original Stated Detection: (Variable A)
2. Strongest Detection Extracted: (Variable B)
3. The Full Detection Controls Text

You must perform two checks:
CHECK 1: Does Variable A exactly match Variable B?
CHECK 2: SEMANTIC VERIFICATION (CRITICAL). Read the actual words in "Full Detection Controls Text". Do they describe the AIAG criteria for the lowest number written next to them? 
- If the text describes a weak control but claims a strong number, Check 2 FAILS (Over-rating).
- If the text describes a strong control but claims a weak number, Check 2 FAILS (Under-rating).

CRITICAL OUTPUT CONSTRAINTS:
You must format your response exactly like one of the following two templates.
TEMPLATE 1 (Pass): "detection controls text matches AIAG detection criteria"
TEMPLATE 2 (Fail): "detection controls text keywords do not match AIAG detection criteria. [State exactly what failed]. Recommended Detection: [Insert the CORRECT AIAG number]"
"""
    
    user_message = (
        f"Original Stated Detection: {clean_original_det}\n"
        f"Strongest Detection Extracted: {pred_det}\n"
        f"Full Detection Controls Text:\n{full_det_text}"
    )
    
    try:
        response = client.chat.completions.create(
            model="gpt-5.4-mini", 
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_message}
            ],
            temperature=0.0
        )
        return response.choices[0].message.content.strip(), response.usage.prompt_tokens, response.usage.completion_tokens
    except Exception as e:
        return f"Error contacting LLM: {str(e)}", 0, 0

def calculate_total_cost(total_input_tokens, total_output_tokens, input_rate_per_m, output_rate_per_m):
    input_cost = (total_input_tokens / 1_000_000) * input_rate_per_m
    output_cost = (total_output_tokens / 1_000_000) * output_rate_per_m
    return input_cost + output_cost, input_cost, output_cost

def shift_merged_cells(ws, insert_col_idx, amount):
    new_ranges = []
    for rng in list(ws.merged_cells.ranges):
        min_col, min_row, max_col, max_row = rng.bounds
        if min_col >= insert_col_idx:
            new_rng = CellRange(min_col=min_col + amount, min_row=min_row, max_col=max_col + amount, max_row=max_row)
            new_ranges.append(new_rng)
        elif min_col < insert_col_idx <= max_col:
            new_rng = CellRange(min_col=min_col, min_row=min_row, max_col=max_col + amount, max_row=max_row)
            new_ranges.append(new_rng)
        else:
            new_ranges.append(rng)
    ws.merged_cells.ranges.clear()
    for rng in new_ranges:
        ws.merged_cells.add(rng)

def inject_modern_css(html_path):
    with open(html_path, 'r', encoding='utf-8') as f:
        html_content = f.read()
        
    parts = html_content.split('<tr')
    target_idx = -1
    for i, part in enumerate(parts):
        if "PROCESS FAILURE MODE AND EFFECTS ANALYSIS" in part.upper():
            target_idx = i
            break
            
    if target_idx > 1:
        for i in range(1, target_idx):
            parts[i] = ' class="hidden-row" ' + parts[i]
    
    html_content = '<tr'.join(parts)
    
    html_content = re.sub(
        r'(<td[^>]*>)(.*?PROCESS FAILURE MODE AND EFFECTS ANALYSIS.*?)(</td>)',
        r'\1<div class="pfmea-main-header">\2</div>\3',
        html_content,
        flags=re.IGNORECASE | re.DOTALL
    )

    modern_css = """
    <style>
        @import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&display=swap');
        body { font-family: 'Inter', sans-serif; background-color: #f4f7f9; color: #334155; margin: 40px; }
        table { border-collapse: collapse; width: 100%; background-color: #ffffff; box-shadow: 0 4px 6px -1px rgba(0,0,0,0.1); font-size: 14px; }
        table, th, td { border: 1px solid #e2e8f0 !important; }
        th, td { padding: 12px 16px !important; line-height: 1.5; }
        .hidden-row { display: none !important; }
        .pfmea-main-header { font-size: 24px !important; font-weight: 700 !important; background-color: #1e3a8a !important; color: #ffffff !important; padding: 16px !important; text-align: center !important; border-radius: 6px; text-transform: uppercase; }
        tr:nth-child(-n+11) td { background-color: #ffffff !important; border: none !important; font-size: 13px; color: #64748b !important; }
        tr:nth-child(12) td, tr:nth-child(13) td { background-color: #0f172a !important; color: #ffffff !important; font-weight: 600 !important; text-transform: uppercase; font-size: 12px; }
        tr:nth-child(n+14):nth-child(even) td { background-color: #f8fafc !important; }
        tr:nth-child(n+14):hover td { background-color: #f1f5f9 !important; transition: background-color 0.2s ease; }
        td[style*="color: #CC0000"], td[style*="color: CC0000"] { color: #b91c1c !important; background-color: #fef2f2 !important; border: 1px solid #fca5a5 !important; font-size: 14px !important; }
    </style>
    """
    
    if '</head>' in html_content:
        html_content = html_content.replace('</head>', f'{modern_css}\n</head>')
    else:
        html_content = modern_css + html_content
        
    with open(html_path, 'w', encoding='utf-8') as f:
        f.write(html_content)

def process_and_audit_pfmea(input_file="pfmea_org.xlsx", output_file="pfmea_audited.xlsx"):
    print("Loading workbook and dynamic detection prompt...")
    wb = openpyxl.load_workbook(input_file)
    ws = wb["PFMEA"]
    
    det_definitions_prompt = build_det_prompt(input_file)

    total_input_tokens, total_output_tokens = 0, 0

    sev_col_idx = 4
    cls_col_idx = 5
    cause_col_idx = 6
    occ_col_idx = 8
    effects_col_idx = 3
    det_controls_col_idx = 9
    det_col_idx = 10
    rpn_col_idx = 11
    
    # Optional columns that might exist
    process_col_idx = None
    sev_zone_col_idx = None

    # Locate Columns dynamically
    for col in range(1, ws.max_column + 1):
        val = str(ws.cell(12, col).value or "").strip().lower()
        if val == "sev": sev_col_idx = col
        elif val == "cls": cls_col_idx = col
        elif "cause" in val or "mechanism" in val: cause_col_idx = col
        elif val == "occ": occ_col_idx = col
        elif val == "effect": effects_col_idx = col
        elif "detection controls" in val: det_controls_col_idx = col
        elif val == "det": det_col_idx = col 
        elif val == "rpn": rpn_col_idx = col 
        elif val == "process": process_col_idx = col
        elif "severity zone" in val or "sev zone" in val: sev_zone_col_idx = col

    # Insert Prediction columns for Sev
    pred_sev_col_idx = sev_col_idx + 1
    reasoning_col_idx = sev_col_idx + 2
    ws.insert_cols(pred_sev_col_idx, 2)
    shift_merged_cells(ws, pred_sev_col_idx, 2)  
    
    if cls_col_idx >= pred_sev_col_idx: cls_col_idx += 2
    if cause_col_idx >= pred_sev_col_idx: cause_col_idx += 2
    if occ_col_idx >= pred_sev_col_idx: occ_col_idx += 2
    if det_controls_col_idx >= pred_sev_col_idx: det_controls_col_idx += 2
    if det_col_idx >= pred_sev_col_idx: det_col_idx += 2
    if rpn_col_idx >= pred_sev_col_idx: rpn_col_idx += 2
    if process_col_idx and process_col_idx >= pred_sev_col_idx: process_col_idx += 2
    if sev_zone_col_idx and sev_zone_col_idx >= pred_sev_col_idx: sev_zone_col_idx += 2

    # Insert Prediction columns for Det
    pred_det_col_idx = det_col_idx + 1
    det_reasoning_col_idx = det_col_idx + 2
    ws.insert_cols(pred_det_col_idx, 2)
    shift_merged_cells(ws, pred_det_col_idx, 2)
    
    if rpn_col_idx >= pred_det_col_idx: rpn_col_idx += 2
    if process_col_idx and process_col_idx >= pred_det_col_idx: process_col_idx += 2
    if sev_zone_col_idx and sev_zone_col_idx >= pred_det_col_idx: sev_zone_col_idx += 2

    # Insert D-P gap
    dp_gap_col_idx = rpn_col_idx + 1
    ws.insert_cols(dp_gap_col_idx, 1)
    shift_merged_cells(ws, dp_gap_col_idx, 1) 
    
    if process_col_idx and process_col_idx >= dp_gap_col_idx: process_col_idx += 1
    if sev_zone_col_idx and sev_zone_col_idx >= dp_gap_col_idx: sev_zone_col_idx += 1

    # Insert New RPN
    new_rpn_col_idx = dp_gap_col_idx + 1
    ws.insert_cols(new_rpn_col_idx, 1)
    shift_merged_cells(ws, new_rpn_col_idx, 1) 
    
    if sev_zone_col_idx and sev_zone_col_idx >= new_rpn_col_idx: sev_zone_col_idx += 1

    # Insert Pred Det Zone & Pred Sev Zone
    pred_det_zone_col_idx = new_rpn_col_idx + 1
    pred_sev_zone_col_idx = new_rpn_col_idx + 2
    ws.insert_cols(pred_det_zone_col_idx, 2)
    shift_merged_cells(ws, pred_det_zone_col_idx, 2)
    
    if sev_zone_col_idx and sev_zone_col_idx >= pred_det_zone_col_idx: sev_zone_col_idx += 2

    # Match Column Widths
    sev_width = ws.column_dimensions[get_column_letter(sev_col_idx)].width or 10
    reasoning_width = 50

    ws.column_dimensions[get_column_letter(cls_col_idx)].width = sev_width
    ws.column_dimensions[get_column_letter(det_col_idx)].width = sev_width
    ws.column_dimensions[get_column_letter(pred_det_zone_col_idx)].width = sev_width
    ws.column_dimensions[get_column_letter(pred_sev_zone_col_idx)].width = sev_width
    
    # Match D-P Gap to Process width
    if process_col_idx:
        process_width = ws.column_dimensions[get_column_letter(process_col_idx)].width
        if process_width:
            ws.column_dimensions[get_column_letter(dp_gap_col_idx)].width = process_width
    else:
        ws.column_dimensions[get_column_letter(dp_gap_col_idx)].width = sev_width
    
    ws.column_dimensions[get_column_letter(cause_col_idx)].width = reasoning_width
    ws.column_dimensions[get_column_letter(effects_col_idx)].width = reasoning_width
    ws.column_dimensions[get_column_letter(det_controls_col_idx)].width = reasoning_width

    headers_config = {
        pred_sev_col_idx: ("pred sev", sev_col_idx, 15),
        reasoning_col_idx: ("sev reasoning", sev_col_idx, reasoning_width),
        pred_det_col_idx: ("pred det", det_col_idx, 15),
        det_reasoning_col_idx: ("det reasoning", det_col_idx, reasoning_width)
    }

    for target_col, (header_title, source_col, width) in headers_config.items():
        ws.cell(12, target_col).value = header_title
        ws.merge_cells(start_row=12, start_column=target_col, end_row=13, end_column=target_col)
        for r in (12, 13):
            source_cell = ws.cell(r, source_col)
            target_cell = ws.cell(r, target_col)
            for prop in ["font", "border", "fill", "number_format"]:
                setattr(target_cell, prop, copy.copy(getattr(source_cell, prop)))
            target_cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        ws.column_dimensions[get_column_letter(target_col)].width = width

    def format_new_column(col_idx, title):
        ws.cell(12, col_idx).value = title
        ws.merge_cells(start_row=12, start_column=col_idx, end_row=13, end_column=col_idx)
        for r in (12, 13):
            source_cell = ws.cell(r, rpn_col_idx)
            target_cell = ws.cell(r, col_idx)
            for prop in ["font", "border", "fill", "number_format"]:
                setattr(target_cell, prop, copy.copy(getattr(source_cell, prop)))
            target_cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)

    format_new_column(dp_gap_col_idx, "D-P gap")
    format_new_column(new_rpn_col_idx, "new RPN")
    ws.column_dimensions[get_column_letter(new_rpn_col_idx)].width = 15
    
    format_new_column(pred_det_zone_col_idx, "pred det zone")
    format_new_column(pred_sev_zone_col_idx, "pred sev zone")

    # Process Data Rows
    for row in range(14, ws.max_row + 1):
        original_sev = ws.cell(row, sev_col_idx).value 
        original_occ = ws.cell(row, occ_col_idx).value
        original_det = ws.cell(row, det_col_idx).value
        
        s_val = extract_number(original_sev)
        o_val = extract_number(original_occ)
        d_val = extract_number(original_det)
        
        # --- CALCULATE ORIGINAL RPN ---
        if s_val is not None and o_val is not None and d_val is not None:
            ws.cell(row, rpn_col_idx).value = s_val * o_val * d_val

        # --- A. EVALUATE SEVERITY ---
        full_effect_text = ws.cell(row, effects_col_idx).value
        pred_sev_cell = ws.cell(row, pred_sev_col_idx)
        sev_reasoning_cell = ws.cell(row, reasoning_col_idx)
        source_sev_cell = ws.cell(row, sev_col_idx)

        for target_cell in [pred_sev_cell, sev_reasoning_cell]:
            for prop in ["border", "fill", "number_format", "font"]:
                setattr(target_cell, prop, copy.copy(getattr(source_sev_cell, prop)))
        pred_sev_cell.alignment = Alignment(horizontal="center", vertical="center")
        sev_reasoning_cell.alignment = Alignment(horizontal="left", vertical="center", wrap_text=True)

        if full_effect_text is not None or original_sev is not None:
            pred_sev, sev_reasoning, s_in, s_out = evaluate_severity_with_llm(
                str(full_effect_text) if full_effect_text else "", original_sev
            )
            total_input_tokens += s_in
            total_output_tokens += s_out
            
            try:
                pred_sev_cell.value = int(pred_sev) if pred_sev else ""
            except ValueError:
                pred_sev_cell.value = pred_sev
                
            sev_reasoning_cell.value = sev_reasoning
            print(f"Auditing Row {row} (Sev) - Predicted: {pred_sev}")
            
            if sev_reasoning and "severity is correct" not in sev_reasoning.lower():
                sev_reasoning_cell.font = Font(color="CC0000")
                pred_sev_cell.font = Font(color="CC0000") 

        # --- B. EVALUATE DETECTION ---
        full_det_text = ws.cell(row, det_controls_col_idx).value
        pred_det_cell = ws.cell(row, pred_det_col_idx)
        det_reasoning_cell = ws.cell(row, det_reasoning_col_idx)
        source_det_cell = ws.cell(row, det_col_idx)

        for target_cell in [pred_det_cell, det_reasoning_cell]:
            for prop in ["border", "fill", "number_format", "font"]:
                setattr(target_cell, prop, copy.copy(getattr(source_det_cell, prop)))
        pred_det_cell.alignment = Alignment(horizontal="center", vertical="center")
        det_reasoning_cell.alignment = Alignment(horizontal="left", vertical="center", wrap_text=True)

        if full_det_text is not None or original_det is not None:
            pred_det_extracted = extract_strongest_detection(full_det_text)
            det_reasoning, d_in, d_out = evaluate_detection_with_llm(full_det_text, original_det, pred_det_extracted, det_definitions_prompt)
            total_input_tokens += d_in
            total_output_tokens += d_out
            
            det_reasoning_cell.value = det_reasoning
            if "keywords do not match" in det_reasoning.lower():
                det_reasoning_cell.font = Font(color="CC0000")
                pred_det_cell.font = Font(color="CC0000") 
                rec_match = re.search(r"Recommended Detection:\s*(\d+)", det_reasoning, re.IGNORECASE)
                if rec_match:
                    pred_det_cell.value = int(rec_match.group(1))
            else:
                try:
                    pred_det_cell.value = int(pred_det_extracted) if pred_det_extracted else ""
                except ValueError:
                    pred_det_cell.value = pred_det_extracted

        # --- C. D-P GAP EVALUATION ---
        dp_gap_cell = ws.cell(row, dp_gap_col_idx)
        source_rpn_cell = ws.cell(row, rpn_col_idx)
        for prop in ["border", "fill", "number_format", "font"]:
            setattr(dp_gap_cell, prop, copy.copy(getattr(source_rpn_cell, prop)))
        dp_gap_cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)

        try:
            sev_val = int(pred_sev_cell.value)
        except (ValueError, TypeError):
            sev_val = 0
        
        if sev_val in [9, 10]:
            try:
                if float(pred_det_cell.value) <= 3:
                    dp_gap_cell.value, dp_gap_cell.font = "no gap", Font(color="000000")
                else:
                    dp_gap_cell.value, dp_gap_cell.font = "Gap identified. Det must be <3", Font(color="CC0000")
            except (ValueError, TypeError):
                dp_gap_cell.value, dp_gap_cell.font = "Gap identified. Det must be <3", Font(color="CC0000")
        elif sev_val > 0:
            dp_gap_cell.value, dp_gap_cell.font = "no gap", Font(color="000000") 

        # --- D. CALCULATE NEW RPN ---
        new_rpn_cell = ws.cell(row, new_rpn_col_idx)
        for prop in ["border", "fill", "number_format", "font"]:
            setattr(new_rpn_cell, prop, copy.copy(getattr(source_rpn_cell, prop)))
        new_rpn_cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        
        p_s_val = extract_number(pred_sev_cell.value)
        p_d_val = extract_number(pred_det_cell.value)
        
        if p_s_val is not None and o_val is not None and p_d_val is not None:
            new_rpn_cell.value = p_s_val * o_val * p_d_val

        # --- E. CALCULATE DET ZONE ---
        pred_det_zone_cell = ws.cell(row, pred_det_zone_col_idx)
        for prop in ["font", "border", "fill", "number_format"]:
            setattr(pred_det_zone_cell, prop, copy.copy(getattr(source_rpn_cell, prop)))
        pred_det_zone_cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)

        if p_s_val is not None and p_d_val is not None:
            pred_det_zone_cell.value = get_det_zone(p_s_val, p_d_val)

        # --- F. CALCULATE SEV ZONE ---
        pred_sev_zone_cell = ws.cell(row, pred_sev_zone_col_idx)
        for prop in ["font", "border", "fill", "number_format"]:
            setattr(pred_sev_zone_cell, prop, copy.copy(getattr(source_rpn_cell, prop)))
        pred_sev_zone_cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)

        if p_s_val is not None and o_val is not None:
            calculated_sev_zone = get_sev_zone(p_s_val, o_val)
            pred_sev_zone_cell.value = calculated_sev_zone
            
            # Check against original Severity Zone column if it exists
            if sev_zone_col_idx is not None:
                orig_sev_zone = extract_number(ws.cell(row, sev_zone_col_idx).value)
                if orig_sev_zone is not None and orig_sev_zone != calculated_sev_zone:
                    pred_sev_zone_cell.font = Font(color="CC0000") # Flag incorrect zones in red

    wb.save(output_file)
    wb.close()
    print(f"\nAuditing complete. Saved to: {output_file}")
    print(f"Total Input Tokens: {total_input_tokens} | Output Tokens: {total_output_tokens}")

if __name__ == "__main__":
    process_and_audit_pfmea()

Loading workbook and dynamic detection prompt...
Auditing Row 14 (Sev) - Predicted: 7
Auditing Row 15 (Sev) - Predicted: 7
Auditing Row 16 (Sev) - Predicted: 9
Auditing Row 17 (Sev) - Predicted: 5
Auditing Row 18 (Sev) - Predicted: 4
Auditing Row 19 (Sev) - Predicted: 8
Auditing Row 20 (Sev) - Predicted: 8
Auditing Row 21 (Sev) - Predicted: 7
Auditing Row 22 (Sev) - Predicted: 9
Auditing Row 23 (Sev) - Predicted: 7

Auditing complete. Saved to: pfmea_audited.xlsx
Total Input Tokens: 15655 | Output Tokens: 662
